# NEXTBUY: Business Insights Bonus

## Part 4: Advanced Business Insights

This notebook contains 8 bonus business questions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

## Load Data

In [ ]:
PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data_engineered.parquet"
fallback_file = Path("full_data_engineered.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError("Processed data not found. Run preprocessing first.")

print(f"Loaded: {full_data.shape}")

---

## Question 9: What products should we recommend to a customer?

In [ ]:
# Simple recommendation based on user's favorite departments
user_id = full_data['user_id'].iloc[0]
user_orders = full_data[full_data['user_id'] == user_id]
user_dept = user_orders['department'].value_counts().head(3).index.tolist()

recommendations = full_data[full_data['department'].isin(user_dept)]['product_name'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(10), recommendations.values[::-1], color='purple')
plt.yticks(range(10), recommendations.index[::-1])
plt.title('Top 10 Recommended Products', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

---

## Question 10: Which items do customers put in their cart first?

In [ ]:
first_items = full_data[full_data['add_to_cart_order'] == 1]
first_items_top = first_items['product_name'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(10), first_items_top.values[::-1], color='steelblue')
plt.yticks(range(10), first_items_top.index[::-1])
plt.title('Top 10 Products Added First to Cart', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

---

## Question 11: Which products have the highest reorder probability?

In [ ]:
product_reorder = full_data.groupby('product_name').agg({
    'reordered': ['sum', 'count']
}).reset_index()
product_reorder.columns = ['product_name', 'reorder_count', 'total_count']
product_reorder['reorder_rate'] = product_reorder['reorder_count'] / product_reorder['total_count']
product_reorder = product_reorder[product_reorder['total_count'] >= 100]
top_reorder = product_reorder.nlargest(10, 'reorder_rate')

plt.figure(figsize=(12, 6))
plt.barh(range(10), top_reorder['reorder_rate'].values[::-1] * 100, color='green')
plt.yticks(range(10), top_reorder['product_name'].values[::-1])
plt.xlabel('Reorder Rate (%)')
plt.title('Top 10 Products with Highest Reorder Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Question 12: Which aisle and department have the most/least products?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

aisle_counts = full_data['aisle'].value_counts()
axes[0].barh(range(10), aisle_counts.head(10).values[::-1], color='teal')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(aisle_counts.head(10).index[::-1])
axes[0].set_title('Top 10 Aisles by Orders')

dept_counts = full_data['department'].value_counts()
axes[1].barh(range(len(dept_counts)), dept_counts.values[::-1], color='orange')
axes[1].set_yticks(range(len(dept_counts)))
axes[1].set_yticklabels(dept_counts.index[::-1])
axes[1].set_title('Departments by Orders')

plt.tight_layout()
plt.show()

---

## Question 13: Are there pairs of aisles that are highly correlated?

In [ ]:
aisle_orders = full_data.groupby(['order_id', 'aisle']).size().unstack(fill_value=0)
aisle_corr = aisle_orders.corr()

corr_pairs = []
for i in range(len(aisle_corr.columns)):
    for j in range(i+1, len(aisle_corr.columns)):
        corr_pairs.append({
            'aisle1': aisle_corr.columns[i],
            'aisle2': aisle_corr.columns[j],
            'correlation': aisle_corr.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs).sort_values('correlation', ascending=False).head(10)

plt.figure(figsize=(12,6))
labels = corr_df['aisle1'] + ' + ' + corr_df['aisle2']
plt.barh(range(10), corr_df['correlation'].values[::-1], color='coral')
plt.yticks(range(10), labels[::-1])
plt.xlabel('Correlation')
plt.title('Top 10 Correlated Aisle Pairs')
plt.tight_layout()
plt.show()

---

## Question 14: What are the most popular product categories ordered together?

In [ ]:
data_clean = full_data.dropna(subset=['department'])
order_depts = data_clean.groupby('order_id')['department'].apply(set).reset_index()

dept_pairs = []
for depts in order_depts['department']:
    if depts and len(depts) > 1:
        dept_list = list(depts)
        for i in range(len(dept_list)):
            for j in range(i+1, len(dept_list)):
                pair = tuple(sorted([dept_list[i], dept_list[j]]))
                dept_pairs.append(pair)

dept_pair_counts = pd.Series(dept_pairs).value_counts().head(10)

plt.figure(figsize=(12,6))
labels = [p[0] + ' + ' + p[1] for p in dept_pair_counts.index]
plt.barh(range(10), dept_pair_counts.values[::-1], color='purple')
plt.yticks(range(10), labels[::-1])
plt.xlabel('Order Count')
plt.title('Top 10 Department Pairs Ordered Together')
plt.tight_layout()
plt.show()

---

## Question 15: Do customers order more cat food in the morning and dog food in the evening?

In [ ]:
cat_food = full_data[full_data['product_name'].str.contains('cat food|cat treats', case=False, na=False)]
dog_food = full_data[full_data['product_name'].str.contains('dog food|dog treats', case=False, na=False)]

cat_morning = cat_food[cat_food['order_hour_of_day'] < 12].shape[0]
cat_evening = cat_food[cat_food['order_hour_of_day'] >= 18].shape[0]
dog_morning = dog_food[dog_food['order_hour_of_day'] < 12].shape[0]
dog_evening = dog_food[dog_food['order_hour_of_day'] >= 18].shape[0]

x = np.arange(2)
plt.figure(figsize=(10,6))
plt.bar(x - 0.175, [cat_morning, dog_morning], 0.35, label='Morning', color='gold')
plt.bar(x + 0.175, [cat_evening, dog_evening], 0.35, label='Evening', color='darkblue')
plt.xticks(x, ['Cat Food', 'Dog Food'])
plt.ylabel('Order Count')
plt.title('Cat Food vs Dog Food: Morning vs Evening')
plt.legend()
plt.tight_layout()
plt.show()

---

## Question 16: Is there a relationship between time since last order and reorder probability?

In [ ]:
valid_orders = full_data[full_data['days_since_prior_order'] >= 0]
reorder_by_days = valid_orders.groupby('days_since_prior_order')['reordered'].mean() * 100

plt.figure(figsize=(12,5))
plt.plot(reorder_by_days.index, reorder_by_days.values, marker='o', color='teal', linewidth=2)
plt.xlabel('Days Since Prior Order')
plt.ylabel('Reorder Rate (%)')
plt.title('Reorder Rate vs Days Since Last Order')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()